In [7]:
import pandas as pd
import numpy as np
import json
import os

def analyze_player_trend(file_path, update_date_str):
    df = pd.read_csv(file_path)
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df = df.sort_values('DateTime')
    update_date = pd.to_datetime(update_date_str)
    
    # Before period is 7 days leading up to the update
    before_update_start = update_date - pd.Timedelta(days=7)
    before_update_end = update_date - pd.Timedelta(days=1)
    before_df = df[(df['DateTime'] >= before_update_start) & (df['DateTime'] <= before_update_end)].copy()
    
    # After period is from the update to the end of the data
    after_df = df[df['DateTime'] >= update_date].copy()
    
    # Averages
    avg_players_before = before_df['Players'].mean()
    avg_players_after = after_df['Players'].mean()
    
    # Percentage change
    if avg_players_before > 0:
        percentage_change = ((avg_players_after - avg_players_before) / avg_players_before) * 100
    else:
        percentage_change = float('inf')
        
    # Slopes
    slope_before = 0
    if not before_df.empty and len(before_df) > 1:
        before_df['DayIndex'] = (before_df['DateTime'] - before_df['DateTime'].min()).dt.days
        try:
            slope_before = np.polyfit(before_df['DayIndex'], before_df['Players'], 1)[0]
        except:
            slope_before = 0

    slope_after = 0
    if not after_df.empty and len(after_df) > 1:
        after_df['DayIndex'] = (after_df['DateTime'] - after_df['DateTime'].min()).dt.days
        try:
            slope_after = np.polyfit(after_df['DayIndex'], after_df['Players'], 1)[0]
        except:
            slope_after = 0

    return avg_players_before, avg_players_after, percentage_change, slope_before, slope_after

# Correcting the file paths
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
data_dir = os.path.join(project_root, 'data:clean')

games = [
    {'name': 'Palworld', 'file': os.path.join(data_dir, 'palworld_2024-12-09_to_2025-01-13.csv'), 'update': '2024-12-23'},
    {'name': 'Warframe', 'file': os.path.join(data_dir, 'warframe_2025-11-26_to_2025-12-31.csv'), 'update': '2025-12-10'},
    {'name': 'Counter-Strike 2', 'file': os.path.join(data_dir, 'counter_strike_2_2024-04-23_to_2024-05-28.csv'), 'update': '2024-05-07'},
    {'name': 'Sea of Thieves', 'file': os.path.join(data_dir, 'sea_of_thieves_2024-10-03_to_2024-11-07.csv'), 'update': '2024-10-17'},
    {'name': 'PUBG: BATTLEGROUNDS', 'file': os.path.join(data_dir, 'pubg__battlegrounds_2025-10-22_to_2025-11-26.csv'), 'update': '2025-11-05'},
    {'name': 'Dead by Daylight', 'file': os.path.join(data_dir, 'dead_by_daylight_2023-11-14_to_2023-12-19.csv'), 'update': '2023-11-28'},
    {'name': "Don't Starve Together", 'file': os.path.join(data_dir, 'don_t_starve_together_2023-04-13_to_2023-05-18.csv'), 'update': '2023-04-27'},
    {'name': 'Deep Rock Galactic', 'file': os.path.join(data_dir, 'deep_rock_galactic_2024-05-30_to_2024-07-04.csv'), 'update': '2024-06-13'},
    {'name': 'Apex Legends', 'file': os.path.join(data_dir, 'apex_legends_2025-01-28_to_2025-03-04.csv'), 'update': '2025-02-11'},
    {'name': 'Destiny 2', 'file': os.path.join(data_dir, 'destiny_2_2024-09-24_to_2024-10-29.csv'), 'update': '2024-10-08'},
    {'name': "No Man's Sky", 'file': os.path.join(data_dir, 'no_man_s_sky_2025-01-15_to_2025-02-19.csv'), 'update': '2025-01-29'},
    {'name': 'HELLDIVERS 2', 'file': os.path.join(data_dir, 'helldivers_2_2025-08-19_to_2025-09-23.csv'), 'update': '2025-09-02'}
]

results = {}
for game in games:
    try:
        avg_before, avg_after, change, slope_before, slope_after = analyze_player_trend(game['file'], game['update'])
        results[game['name']] = {
            'avg_before': avg_before,
            'avg_after': avg_after,
            'percentage_change': change,
            'slope_before': slope_before,
            'slope_after': slope_after
        }
    except Exception as e:
        results[game['name']] = {'error': str(e)}

print(json.dumps(results, indent=2))

{
  "Palworld": {
    "avg_before": -1.1205206584806613,
    "avg_after": 0.7559436791404085,
    "percentage_change": Infinity,
    "slope_before": 0.03332970008736834,
    "slope_after": -0.017032274010149707
  },
  "Warframe": {
    "avg_before": -1.0786904079031376,
    "avg_after": 0.6725321474886229,
    "percentage_change": Infinity,
    "slope_before": 0.06949758095407531,
    "slope_after": -0.08215141493241003
  },
  "Counter-Strike 2": {
    "avg_before": -0.9846901408824316,
    "avg_after": 0.20903377360777595,
    "percentage_change": Infinity,
    "slope_before": 0.35089783652095424,
    "slope_after": -0.054310754337417756
  },
  "Sea of Thieves": {
    "avg_before": 0.01751936413303516,
    "avg_after": 0.11992796086401744,
    "percentage_change": 584.5451692957101,
    "slope_before": -0.0042053070479358655,
    "slope_after": -0.11716325041063157
  },
  "PUBG: BATTLEGROUNDS": {
    "avg_before": -0.759161126226089,
    "avg_after": 0.2629811458178989,
    "percentag

In [8]:
import json

# The 'results' dictionary is from the previous cell execution
markdown_output = "## Player Trend Analysis: Before vs. After Update\n\n"
markdown_output += "| Game | Avg Players Before | Avg Players After | Pct Change (%) | Slope Before | Slope After |\n"
markdown_output += "|------|--------------------|-------------------|----------------|--------------|-------------|\n"

for game_name, data in results.items():
    if 'error' in data:
        row = f"| {game_name} | Error | Error | Error | Error | Error |\n"
    else:
        avg_before = f"{data['avg_before']:.2f}" if data['avg_before'] is not None else "N/A"
        avg_after = f"{data['avg_after']:.2f}" if data['avg_after'] is not None else "N/A"
        percentage_change = f"{data['percentage_change']:.2f}" if data['percentage_change'] is not None else "N/A"
        slope_before = f"{data['slope_before']:.2f}" if data['slope_before'] is not None else "N/A"
        slope_after = f"{data['slope_after']:.2f}" if data['slope_after'] is not None else "N/A"
        row = f"| {game_name} | {avg_before} | {avg_after} | {percentage_change} | {slope_before} | {slope_after} |\n"
    markdown_output += row

# Save to a file to be appended to analysis.md
output_file = "../outputs/before_after_trend_analysis.md"
with open(output_file, "w") as f:
    f.write(markdown_output)

print(f"Trend analysis results saved to {output_file}")
print(markdown_output)

Trend analysis results saved to ../outputs/before_after_trend_analysis.md
## Player Trend Analysis: Before vs. After Update

| Game | Avg Players Before | Avg Players After | Pct Change (%) | Slope Before | Slope After |
|------|--------------------|-------------------|----------------|--------------|-------------|
| Palworld | -1.12 | 0.76 | inf | 0.03 | -0.02 |
| Warframe | -1.08 | 0.67 | inf | 0.07 | -0.08 |
| Counter-Strike 2 | -0.98 | 0.21 | inf | 0.35 | -0.05 |
| Sea of Thieves | 0.02 | 0.12 | 584.55 | -0.00 | -0.12 |
| PUBG: BATTLEGROUNDS | -0.76 | 0.26 | inf | 0.05 | -0.10 |
| Dead by Daylight | -0.36 | 0.22 | inf | 0.15 | -0.09 |
| Don't Starve Together | -1.00 | 0.65 | inf | -0.03 | -0.04 |
| Deep Rock Galactic | -1.03 | 0.70 | inf | -0.03 | -0.08 |
| Apex Legends | -0.48 | 0.60 | inf | 0.14 | -0.08 |
| Destiny 2 | -0.78 | 0.58 | inf | 0.04 | -0.10 |
| No Man's Sky | -1.14 | 0.71 | inf | 0.00 | 0.00 |
| HELLDIVERS 2 | 0.81 | 0.19 | -76.61 | -0.06 | -0.12 |

